# LSTM Autoencoder for Anomaly Detection

### Library Imports

In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path("..").resolve()
sys.path.append(str(ROOT / "src"))

from data.lstm_autoencoder_preprocessing import preprocess_autoencoder_data
from features.lstm_autoencoder_features import build_windowed_splits
from evaluation.lstm_autoencoder_evaluation import build_anomaly_table, evaluate_autoencoder, plot_error_distribution, plot_training_history, predict_window_anomaly
from training.lstm_autoencoder_training import create_dataloaders, get_device, train_autoencoder
from models.lstm_autoencoder import LSTMAutoencoder

## Data Preprocessing

### Load the Data

In [ ]:
turbine_data : pd.DataFrame = pd.read_csv("..\\data\\raw\\turbine_5yr_complex_data.csv")

### Preprocess Data

In [ ]:
prepped = preprocess_autoencoder_data(turbine_data, exclude_columns=["fault_label"])
prepped.train.head()

## Feature Engineering

In [ ]:
windowed = build_windowed_splits(prepped.train, prepped.val, prepped.test, prepped.feature_columns, window_size=60, step_size=5)
loaders = create_dataloaders(windowed, batch_size=64)

## LSTM Autoencoder

### Instantiate

In [ ]:
device = get_device()
model = LSTMAutoencoder(input_size=windowed.train.windows.shape[2], hidden_size=64, latent_size=32, num_layers=2, dropout=0.1).to(device)

### Train

In [ ]:
train_result = train_autoencoder(model, loaders.train, loaders.val, device=device)

### Evaluation

In [ ]:
eval_result = evaluate_autoencoder(model, loaders.test, windowed.test, device=device, label_series=prepped.test.get("fault_label")) #type: ignore
eval_result.metrics

## Visualising Results

In [ ]:
plot_training_history(train_result.history) #type: ignore
plot_error_distribution(eval_result.errors, eval_result.threshold)

## Identified Anomalies

In [ ]:
anomaly_table = build_anomaly_table(turbine_data, windowed.test, eval_result.errors, eval_result.threshold)
anomaly_table